In [1]:
"""Imports"""
%load_ext autoreload
%autoreload 2

import sys
import os

root = os.path.abspath("..")
if root not in sys.path:
    sys.path.insert(0, root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data

In [2]:
""" Load raw data
"""
df_raw = load_raw_data()
print(f"Raw data loaded: {df_raw.shape}")

Raw data loaded: (5001, 21)


In [3]:
df_raw.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,CUST-00005,Male,0,Yes,Yes,53,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.29,1553.16,No
1,CUST-00001,Male,0,No,Yes,61,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Credit card,87.04,5303.45,No
2,CUST-00002,Female,0,No,No,68,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card,28.63,1944.52,No
3,CUST-00003,Male,0,No,NaN,62,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card,27.85,1736.17,No
4,CUST-00004,Male,1,Yes,No,1,Yes,Yes,DSL,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,67.92,62.16,Yes


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5001 entries, 0 to 5000
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        5001 non-null   object 
 1   gender             5001 non-null   object 
 2   senior_citizen     5001 non-null   int64  
 3   partner            5001 non-null   object 
 4   dependents         4801 non-null   object 
 5   tenure_months      5001 non-null   int64  
 6   phone_service      5001 non-null   object 
 7   multiple_lines     5001 non-null   object 
 8   internet_service   5001 non-null   object 
 9   online_security    5001 non-null   object 
 10  online_backup      5001 non-null   object 
 11  device_protection  5001 non-null   object 
 12  tech_support       5001 non-null   object 
 13  streaming_tv       5001 non-null   object 
 14  streaming_movies   5001 non-null   object 
 15  contract_type      5001 non-null   object 
 16  paperless_billing  5001 

In [5]:
df_raw.columns

Index(['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents',
       'tenure_months', 'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract_type',
       'paperless_billing', 'payment_method', 'monthly_charges',
       'total_charges', 'churn'],
      dtype='object')

In [6]:
""" Convert total charges column from object dtype to float dtype
"""

df = df_raw.copy()

df['total_charges'].isna().sum()
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
df['total_charges'].values.dtype

dtype('float64')

In [7]:
df[['senior_citizen']].value_counts()

senior_citizen
0                 4276
1                  725
Name: count, dtype: int64

In [8]:
df['senior_citizen'] = df['senior_citizen'].map({0: "No", 1: "Yes"})

In [9]:
df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,CUST-00005,Male,No,Yes,Yes,53,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.29,1553.16,No
1,CUST-00001,Male,No,No,Yes,61,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Credit card,87.04,5303.45,No
2,CUST-00002,Female,No,No,No,68,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card,28.63,1944.52,No
3,CUST-00003,Male,No,No,NaN,62,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card,27.85,1736.17,No
4,CUST-00004,Male,Yes,Yes,No,1,Yes,Yes,DSL,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,67.92,62.16,Yes


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5001 entries, 0 to 5000
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        5001 non-null   object 
 1   gender             5001 non-null   object 
 2   senior_citizen     5001 non-null   object 
 3   partner            5001 non-null   object 
 4   dependents         4801 non-null   object 
 5   tenure_months      5001 non-null   int64  
 6   phone_service      5001 non-null   object 
 7   multiple_lines     5001 non-null   object 
 8   internet_service   5001 non-null   object 
 9   online_security    5001 non-null   object 
 10  online_backup      5001 non-null   object 
 11  device_protection  5001 non-null   object 
 12  tech_support       5001 non-null   object 
 13  streaming_tv       5001 non-null   object 
 14  streaming_movies   5001 non-null   object 
 15  contract_type      5001 non-null   object 
 16  paperless_billing  5001 

In [11]:
""" Impute missing values 
Pass 1: Apply business logic
Pass 2: Statistical approach
        Median, Mean, Mode
"""
mask1 = df['total_charges'].isna()
df[mask1]['tenure_months'].value_counts()

tenure_months
0    64
Name: count, dtype: int64

In [12]:
df.loc[mask1, 'total_charges'] = 0.0

In [13]:
df.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,CUST-00005,Male,No,Yes,Yes,53,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.29,1553.16,No
1,CUST-00001,Male,No,No,Yes,61,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Credit card,87.04,5303.45,No
2,CUST-00002,Female,No,No,No,68,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card,28.63,1944.52,No
3,CUST-00003,Male,No,No,NaN,62,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card,27.85,1736.17,No
4,CUST-00004,Male,Yes,Yes,No,1,Yes,Yes,DSL,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,67.92,62.16,Yes


In [14]:
df['total_charges'].isna().sum()

np.int64(0)

In [15]:
mean = df['total_charges'].mean()

In [16]:
mask2 = df['total_charges'].isna()
df.loc[mask2, 'total_charges'] = mean

In [17]:
""" Impute categorical missing columns """

categorical_cols = [
    'dependents', 'payment_method'
]

df['payment_method'].isna().sum()

np.int64(150)

In [18]:
mode = df['dependents'].mode()[0]

In [19]:
mask3 = df['dependents'].isna()
df.loc[mask3, 'dependents'] = mode

In [20]:
df[mask3]

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
3,CUST-00003,Male,No,No,No,62,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card,27.85,1736.17,No
72,CUST-00072,Female,Yes,Yes,No,2,Yes,Yes,Fiber optic,No,...,No,Yes,Yes,Yes,Month-to-month,Yes,NaN,101.05,206.03,No
157,CUST-00157,Male,No,No,No,68,Yes,No,Fiber optic,Yes,...,No,No,Yes,No,One year,Yes,Mailed check,92.34,6282.56,No
183,CUST-00183,Female,No,Yes,No,56,Yes,No,Fiber optic,No,...,No,Yes,No,No,One year,Yes,Mailed check,91.65,5136.83,No
193,CUST-00193,Male,No,Yes,No,33,No,No phone service,Fiber optic,Yes,...,No,No,Yes,No,Month-to-month,No,Electronic check,83.32,2747.77,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4969,CUST-04969,Male,No,Yes,No,48,Yes,Yes,DSL,No,...,Yes,No,No,Yes,Month-to-month,Yes,Mailed check,70.08,3356.84,Yes
4975,CUST-04975,Female,No,No,No,48,Yes,No,Fiber optic,Yes,...,No,Yes,Yes,No,Month-to-month,Yes,NaN,98.67,4740.46,Yes
4976,CUST-04976,Female,No,Yes,No,53,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,28.05,1491.55,No
4990,CUST-04990,Female,No,Yes,No,40,Yes,No,Fiber optic,Yes,...,No,No,No,Yes,Month-to-month,No,Mailed check,93.50,3749.51,No


In [21]:
""" Remove whitespaces from the column values """
df['multiple_lines'].value_counts().keys()

Index(['Yes', 'No', 'No phone service'], dtype='object', name='multiple_lines')

In [22]:
df['multiple_lines'] = df['multiple_lines'].str.strip()

In [23]:
var = '  No  '
var.strip()

'No'

In [24]:
cols = df.select_dtypes(include="object").columns
cols

Index(['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents',
       'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract_type',
       'paperless_billing', 'payment_method', 'churn'],
      dtype='object')

In [25]:
for col in cols:
    df[col] = df[col].str.strip()

In [26]:
df.columns = (df.columns
 .str.strip()
 .str.lower()
 .str.replace(' ', '_', regex=False)
 .str.replace('-', '_', regex=False))
df.columns

Index(['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents',
       'tenure_months', 'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract_type',
       'paperless_billing', 'payment_method', 'monthly_charges',
       'total_charges', 'churn'],
      dtype='object')

In [27]:
""" 
Removal of duplicate rows
"""
mask4 = df.duplicated()
df = df.drop_duplicates(keep='first')

df.duplicated().sum()

np.int64(0)

In [28]:
from src.cleaner import run_full_cleaning_pipeline

df_clean, clean_audit = run_full_cleaning_pipeline(df_raw)

In [29]:
df_clean.head()

,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,CUST-00005,Male,No,Yes,Yes,53,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Electronic check,29.29,1553.16,No
1,CUST-00001,Male,No,No,Yes,61,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Credit card,87.04,5303.45,No
2,CUST-00002,Female,No,No,No,68,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card,28.63,1944.52,No
3,CUST-00003,Male,No,No,No,62,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card,27.85,1736.17,No
4,CUST-00004,Male,Yes,Yes,No,1,Yes,Yes,DSL,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,67.92,62.16,Yes
